# DX 704 Week 9 Project

This week's project will build an email spam classifier based on the Enron email data set.
You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

--2026-03-20 18:01:50--  https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip [following]
--2026-03-20 18:01:50--  https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15642124 (15M) [application/zip]
Saving to: ‘enron_spam_data.zip’

enron_spam_data.zip 100%[===================>]  14.92M  46.0MB/s    in 0.3s    

2026-03-20 18:01:51 (46.0 MB/s) - ‘enron_spa

In [3]:
import pandas as pd

In [4]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [5]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the Subject and Message columns are relevant sources of text data.
For each email, you should count the number of repetitions of each feature present.
The auto-grader will assume that you are using a multinomial distribution in the following problems.

In [6]:
# using bag of words

import re
import json
from collections import Counter

# simple tokenizer:
# - lowercase
# - normalize obvious spam markers
# - keep alphabetic tokens and a few special placeholders
WORD_RE = re.compile(r"[a-z$']+")

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()

    # normalize common patterns
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " EMAILTOKEN ", text)
    text = re.sub(r"\$\d+(?:\.\d+)?", " MONEYTOKEN ", text)
    text = re.sub(r"\b\d+(?:\.\d+)?\b", " NUMTOKEN ", text)

    return text

def tokenize(text):
    text = normalize_text(text)
    return WORD_RE.findall(text)

def extract_features(row):
    feats = Counter()

    subject = "" if pd.isna(row["Subject"]) else str(row["Subject"])
    message = "" if pd.isna(row["Message"]) else str(row["Message"])

    # tokenize separately so subject can get its own prefix
    subj_tokens = tokenize(subject)
    body_tokens = tokenize(message)

    # 1) unigram counts from subject
    for tok in subj_tokens:
        feats[f"subj:{tok}"] += 1

    # 2) unigram counts from body
    for tok in body_tokens:
        feats[f"body:{tok}"] += 1

    # 3) bigrams from subject
    for i in range(len(subj_tokens) - 1):
        feats[f"subj_bigram:{subj_tokens[i]}_{subj_tokens[i+1]}"] += 1

    # 4) bigrams from body
    for i in range(len(body_tokens) - 1):
        feats[f"body_bigram:{body_tokens[i]}_{body_tokens[i+1]}"] += 1

    # 5) a few structural / spammy count features
    full_text = f"{subject} {message}"
    full_text_lower = full_text.lower()

    feats["meta:exclamations"] = full_text.count("!")
    feats["meta:question_marks"] = full_text.count("?")
    feats["meta:has_html"] = int("<html" in full_text_lower or "<body" in full_text_lower)
    feats["meta:url_count"] = len(re.findall(r"https?://\S+|www\.\S+", full_text_lower))
    feats["meta:email_count"] = len(re.findall(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", full_text_lower))
    feats["meta:dollar_count"] = full_text.count("$")
    feats["meta:digit_count"] = sum(ch.isdigit() for ch in full_text)
    feats["meta:uppercase_count"] = sum(ch.isupper() for ch in full_text if ch.isalpha())

    # remove zero entries just in case
    feats = {k: int(v) for k, v in feats.items() if int(v) != 0}
    return feats

Assign a row to the test data set if `Message ID % 30 == 0` and assign it to the training data set otherwise.
Write two files, "train-features.tsv" and "test-features.tsv" with two columns, Message ID and features_json.
The features_json column should contain a JSON dictionary where the keys are your feature names and the values are integer feature values.
This will give us a sparse feature representation.


In [7]:
feature_rows = []

for _, row in enron_spam_data.iterrows():
    msg_id = int(row["Message ID"])
    feats = extract_features(row)

    feature_rows.append({
        "Message ID": msg_id,
        "features_json": json.dumps(feats, sort_keys=True)
    })

features_df = pd.DataFrame(feature_rows)

train_features = features_df[features_df["Message ID"] % 30 != 0].copy()
test_features  = features_df[features_df["Message ID"] % 30 == 0].copy()

train_features.to_csv("train-features.tsv", sep="\t", index=False)
test_features.to_csv("test-features.tsv", sep="\t", index=False)

train_features.head(), test_features.head(), train_features.shape, test_features.shape

(   Message ID                                      features_json
 1           1  {"body:'": 3, "body:a": 6, "body:about": 1, "b...
 2           2  {"body:calpine": 1, "body:daily": 1, "body:doc...
 3           3  {"body:accounting": 1, "body:already": 2, "bod...
 4           4  {"body:a": 5, "body:allen": 3, "body:allocated...
 5           5  {"body:active": 1, "body:advice": 1, "body:als...,
      Message ID                                      features_json
 0             0  {"subj:christmas": 1, "subj:farm": 1, "subj:pi...
 30           30  {"body:'": 1, "body:a": 1, "body:all": 1, "bod...
 60           60  {"body:any": 1, "body:as": 1, "body:backup": 1...
 90           90  {"body:enron": 2, "body:hpl": 2, "body:ic": 1,...
 120         120  {"body:a": 2, "body:above": 1, "body:according...,
 (32592, 2),
 (1124, 2))

Submit "train-features.tsv" and "test-features.tsv" in Gradescope.

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

## Part 3: Compute Conditional Probabilities

Based on your training data, compute appropriate conditional probabilities for use with naïve Bayes.
Use of additive smoothing with $\alpha=1$ to avoid zeros.


In [8]:
# load training features
train_features = pd.read_csv("train-features.tsv", sep="\t")

# merge with labels
train_df = train_features.merge(
    enron_spam_data[["Message ID", "Spam/Ham"]],
    on="Message ID",
    how="left"
)

In [9]:
spam_counts = Counter()
ham_counts = Counter()

for _, row in train_df.iterrows():
    feats = json.loads(row["features_json"])
    label = row["Spam/Ham"]

    if label == "spam":
        spam_counts.update(feats)
    else:
        ham_counts.update(feats)

In [10]:
# total counts per class
total_spam = sum(spam_counts.values())
total_ham = sum(ham_counts.values())

# vocabulary = all unique features
vocab = set(spam_counts.keys()) | set(ham_counts.keys())
V = len(vocab)

print("Vocab size:", V)

Vocab size: 1535058


In [11]:
# laplace

model_rows = []

for feat in vocab:
    spam_count = spam_counts.get(feat, 0)
    ham_count = ham_counts.get(feat, 0)

    p_spam = (spam_count + 1) / (total_spam + V)
    p_ham  = (ham_count + 1) / (total_ham + V)

    model_rows.append({
        "feature": feat,
        "p_feature_given_spam": p_spam,
        "p_feature_given_ham": p_ham
    })

model_df = pd.DataFrame(model_rows)

Save the conditional probabilities in a file "feature-probabilities.tsv" with columns feature, ham_probability and spam_probability.

In [13]:
model_rows = []

for feat in vocab:
    spam_count = spam_counts.get(feat, 0)
    ham_count = ham_counts.get(feat, 0)

    spam_prob = (spam_count + 1) / (total_spam + V)
    ham_prob  = (ham_count + 1) / (total_ham + V)

    model_rows.append({
        "feature": feat,
        "ham_probability": ham_prob,
        "spam_probability": spam_prob
    })

feature_probs_df = pd.DataFrame(model_rows)

# save with required filename + column names
feature_probs_df.to_csv("feature-probabilities.tsv", sep="\t", index=False)

feature_probs_df.head(), feature_probs_df.shape

(                        feature  ham_probability  spam_probability
 0      body_bigram:cap_contract     1.878843e-07      1.157135e-07
 1     body_bigram:keyboards_the     9.394217e-08      2.314270e-07
 2          body_bigram:game_now     9.394217e-08      2.314270e-07
 3          body_bigram:his_mind     1.409133e-06      5.785676e-07
 4  body_bigram:economics_majors     1.878843e-07      1.157135e-07,
 (1535058, 3))

In [14]:
print(feature_probs_df.isna().sum())
print(feature_probs_df.describe())

feature             0
ham_probability     0
spam_probability    0
dtype: int64
       ham_probability  spam_probability
count     1.535058e+06      1.535058e+06
mean      6.514412e-07      6.514412e-07
std       6.048659e-05      4.540101e-05
min       9.394217e-08      1.157135e-07
25%       9.394217e-08      1.157135e-07
50%       1.878843e-07      2.314270e-07
75%       2.818265e-07      2.314270e-07
max       6.864467e-02      4.993929e-02


Submit "feature-probabilities.tsv" in Gradescope.

## Part 4: Implement a Naïve Bayes Classifier

Implement a naïve Bayes classifier based on your previous feature probabilities.

In [21]:
train_features = pd.read_csv("train-features.tsv", sep="\t")

In [22]:
def predict_proba(feats):
    log_spam = log_p_spam
    log_ham  = log_p_ham

    for f, count in feats.items():
        if f in spam_probs:
            log_spam += count * np.log(spam_probs[f])
            log_ham  += count * np.log(ham_probs[f])

    # numerical stability trick
    max_log = max(log_spam, log_ham)
    log_spam -= max_log
    log_ham  -= max_log

    spam_score = np.exp(log_spam)
    ham_score  = np.exp(log_ham)

    total = spam_score + ham_score

    p_spam = spam_score / total
    p_ham  = ham_score / total

    return p_ham, p_spam

In [23]:
train_pred_rows = []

for _, row in train_features.iterrows():
    msg_id = row["Message ID"]
    feats = json.loads(row["features_json"])

    p_ham, p_spam = predict_proba(feats)

    train_pred_rows.append({
        "Message ID": msg_id,
        "ham": p_ham,
        "spam": p_spam
    })

train_pred_df = pd.DataFrame(train_pred_rows)

Save your prediction probabilities to "train-predictions.tsv" with columns Message ID, ham and spam.

In [24]:
train_pred_df.to_csv("train-predictions.tsv", sep="\t", index=False)

train_pred_df.head(), train_pred_df.shape

(   Message ID  ham          spam
 0           1  1.0  0.000000e+00
 1           2  1.0  5.916145e-25
 2           3  1.0  0.000000e+00
 3           4  1.0  0.000000e+00
 4           5  1.0  7.184302e-93,
 (32592, 3))

In [25]:
# sanity checks

# probabilities should sum to ~1
(train_pred_df["ham"] + train_pred_df["spam"]).head()

# no NaNs
train_pred_df.isna().sum()

# values between 0 and 1
train_pred_df.describe()

,Message ID,ham,spam
count,32592.000000,3.259200e+04,3.259200e+04
mean,16857.931087,4.899454e-01,5.100546e-01
std,9733.080174,4.989623e-01,4.989623e-01
min,1.000000,0.000000e+00,0.000000e+00
25%,8428.750000,1.416960e-99,5.812680e-115
50%,16857.500000,8.138507e-08,9.999999e-01
75%,25286.250000,1.000000e+00,1.000000e+00
max,33715.000000,1.000000e+00,1.000000e+00


Submit "train-predictions.tsv" in Gradescope.

## Part 5: Predict Spam Probability for Test Data

Use your previous classifier to predict spam probability for the test data.

In [27]:
# same as above but running on test data

# load test features
test_features = pd.read_csv("test-features.tsv", sep="\t")

test_pred_rows = []

for _, row in test_features.iterrows():
    msg_id = row["Message ID"]
    feats = json.loads(row["features_json"])

    p_ham, p_spam = predict_proba(feats)

    test_pred_rows.append({
        "Message ID": msg_id,
        "ham": p_ham,
        "spam": p_spam
    })

test_pred_df = pd.DataFrame(test_pred_rows)

Save your prediction probabilities in "test-predictions.tsv" with the same columns as "train-predictions.tsv".

In [28]:
test_pred_df.to_csv("test-predictions.tsv", sep="\t", index=False)

test_pred_df.head(), test_pred_df.shape

(   Message ID       ham           spam
 0           0  0.578277   4.217230e-01
 1          30  1.000000  7.237301e-222
 2          60  1.000000   1.208054e-24
 3          90  1.000000   1.086229e-57
 4         120  1.000000   0.000000e+00,
 (1124, 3))

Submit "test-predictions.tsv" in Gradescope.

## Part 6: Construct ROC Curve

For every probability threshold from 0.01 to .99 in increments of 0.01, compute the false and true positive rates from the test data using the spam class for positives.
That is, if the predicted spam probability is greater than or equal to the threshold, predict spam.

In [29]:
# load test probabilities
test_pred_df = pd.read_csv("test-predictions.tsv", sep="\t")

# get true labels for the test set
test_labels = enron_spam_data[enron_spam_data["Message ID"] % 30 == 0][["Message ID", "Spam/Ham"]].copy()

# predictions + truth
roc_df_input = test_pred_df.merge(test_labels, on="Message ID", how="inner")

roc_rows = []

for i in range(1, 100):
    threshold = i / 100.0

    # spam if spam probability >= threshold
    pred_spam = roc_df_input["spam"] >= threshold
    actual_spam = roc_df_input["Spam/Ham"] == "spam"

    tp = ((pred_spam) & (actual_spam)).sum()
    fp = ((pred_spam) & (~actual_spam)).sum()
    tn = ((~pred_spam) & (~actual_spam)).sum()
    fn = ((~pred_spam) & (actual_spam)).sum()

    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    roc_rows.append({
        "threshold": threshold,
        "false_positive_rate": fpr,
        "true_positive rate": tpr
    })

roc_df = pd.DataFrame(roc_rows)

Save this data in a file "roc.tsv" with columns threshold, false_positive_rate and true_positive rate.

In [30]:
roc_df.to_csv("roc.tsv", sep="\t", index=False)

roc_df.head(), roc_df.shape

(   threshold  false_positive_rate  true_positive rate
 0       0.01             0.016304            0.993007
 1       0.02             0.014493            0.993007
 2       0.03             0.014493            0.993007
 3       0.04             0.012681            0.993007
 4       0.05             0.012681            0.991259,
 (99, 3))

Submit "roc.tsv" in Gradescope.

## Part 7: Signup for Gemini API Key

Create a free Gemini API key at https://aistudio.google.com/app/api-keys.
You will need to do this with a personal Google account - it will not work with your BU Google account.
This will not incur any charges unless you configure billing information for the key.

You will be asked to start a Gemini free trial for week 11.
This will not incur any charges unless you exceed expected usage by an order of magnitude.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [ ]:
content = (
    "Acknowledgments\n\n"
    "I used the DX704 course-provided code examples as a reference when developing "
    "the code for this assignment.\n\n"
    "I also used reddit and youtube.\n"
    "Last, my fiance works at google and I asked him to check my code and help me where I got stuck."
)

with open("acknowledgments.txt", "w") as f:
    f.write(content)